In [ ]:
# -*- coding: utf-8 -*-
"""Mental Health Analysis with GPT-3.5 Suggestions"""

import sys
import os
import re
import numpy as np
import torch
import torch.nn as nn
from transformers import (
    DistilBertTokenizer, DistilBertModel,
    BertTokenizer, BertModel
)
from typing import Dict, List, Optional
from openai import OpenAI
import dotenv
from dotenv import load_dotenv

# Add safe globals for model loading
import torch.serialization
from transformers import DistilBertTokenizer
torch.serialization.add_safe_globals([DistilBertTokenizer])

# Load environment variables
load_dotenv()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

# Initialize OpenAI client
client = OpenAI(api_key=OPENAI_API_KEY)

# Set device
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

# ==============================================
# MODEL ARCHITECTURES
# ==============================================

class EmotionModel(nn.Module):
    def __init__(self, n_classes):
        super().__init__()
        self.distilbert = DistilBertModel.from_pretrained('distilbert-base-uncased')
        self.dropout = nn.Dropout(0.3)
        self.classifier = nn.Linear(self.distilbert.config.hidden_size, n_classes)

    def forward(self, input_ids, attention_mask):
        outputs = self.distilbert(input_ids=input_ids, attention_mask=attention_mask)
        pooled_output = outputs.last_hidden_state[:, 0]
        pooled_output = self.dropout(pooled_output)
        logits = self.classifier(pooled_output)
        return logits

class MHModel(nn.Module):
    def __init__(self, n_classes, n_emotions_plus_vader):
        super().__init__()
        self.bert = BertModel.from_pretrained('bert-base-uncased')
        bert_hidden = self.bert.config.hidden_size

        self.bilstm = nn.LSTM(
            input_size=bert_hidden,
            hidden_size=128,
            num_layers=2,
            bidirectional=True,
            batch_first=True,
            dropout=0.2
        )

        self.emotion_proj = nn.Sequential(
            nn.Linear(n_emotions_plus_vader, 128),
            nn.ReLU(),
            nn.Dropout(0.2)
        )

        self.classifier = nn.Sequential(
            nn.Linear(256 + 128, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, n_classes)
        )

    def forward(self, input_ids, attention_mask, additional_features):
        bert_output = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        sequence_output = bert_output.last_hidden_state

        lstm_output, _ = self.bilstm(sequence_output)
        lstm_pooled = lstm_output[:, 0]

        emotion_features = self.emotion_proj(additional_features)

        combined = torch.cat([lstm_pooled, emotion_features], dim=1)

        logits = self.classifier(combined)
        return logits

# ==============================================
# SUGGESTION GENERATOR CLASS
# ==============================================

class GPT3SuggestionGenerator:
    def __init__(
        self,
        emotion_model_path: str,
        mh_model_path: str,
        gpt_model: str = "gpt-3.5-turbo",
        temperature: float = 0.7,
        max_tokens: int = 150
    ):
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        print(f"\nInitializing models on device: {self.device}")

        # Load emotion model
        try:
            print("\nLoading emotion model...")
            self.emotion_model, self.emotion_tokenizer, self.emotion_cols, self.emotion_thresholds = \
                self._load_emotion_model(emotion_model_path)
            print(f"✅ Emotion model loaded successfully")
            print(f"Detecting emotions: {self.emotion_cols}")

            # Display thresholds safely
            if isinstance(self.emotion_thresholds, dict):
                print(f"Sample thresholds: {dict(list(self.emotion_thresholds.items())[:3])}...")
            else:
                print(f"Thresholds: {self.emotion_thresholds[:3]}...")
        except Exception as e:
            print(f"❌ Error loading emotion model: {str(e)}")
            raise

        # Load mental health model
        try:
            print("\nLoading mental health model...")
            self.mh_model, self.mh_tokenizer, self.label_encoder, self.remaining_classes = \
                self._load_mh_model(mh_model_path)
            print(f"✅ Mental health model loaded successfully")
            print(f"Possible MH statuses: {self.remaining_classes}")
        except Exception as e:
            print(f"❌ Error loading mental health model: {str(e)}")
            raise

        self.gpt_model = gpt_model
        self.temperature = temperature
        self.max_tokens = max_tokens

        self.system_prompt = """You are a compassionate mental health assistant that provides
        supportive suggestions based on detected emotions and mental health status. Your responses
        should be:
        1. Empathetic and non-judgmental
        2. Practical and actionable
        3. Appropriate for the detected mental health condition
        4. Never claim to be a doctor or provide medical advice
        5. Always include crisis resources when suicidal ideation is detected
        """

        self.crisis_resources = {
            'US': "National Suicide Prevention Lifeline: 988",
            'CA': "Canada Suicide Prevention Service: 1-833-456-4566",
            'UK': "Samaritans: 116 123",
            'AU': "Lifeline Australia: 13 11 14",
            'default': "International Association for Suicide Prevention: https://www.iasp.info/resources/Crisis_Centres/"
        }

    def _load_emotion_model(self, model_path: str):
        """Load emotion model with proper threshold handling"""
        print(f"Loading from: {model_path}")

        checkpoint = torch.load(model_path, map_location=self.device, weights_only=False)

        emotion_cols = checkpoint['emotion_cols']
        print(f"Found {len(emotion_cols)} emotion categories")

        thresholds = checkpoint['thresholds']

        # Handle both dict and array threshold formats
        if isinstance(thresholds, dict):
            print("Converting threshold dictionary to array...")
            thresholds = np.array([thresholds[col] for col in emotion_cols])
        elif isinstance(thresholds, (list, np.ndarray)):
            thresholds = np.array(thresholds)
        else:
            raise ValueError("Thresholds must be either dict or array-like")

        # Verify shape
        if len(thresholds) != len(emotion_cols):
            raise ValueError(f"Thresholds length ({len(thresholds)}) doesn't match emotion columns ({len(emotion_cols)})")

        model = EmotionModel(len(emotion_cols)).to(self.device)

        state_dict = checkpoint['model_state_dict']
        fixed_state_dict = {}
        for k, v in state_dict.items():
            if k.startswith('bert.'):
                fixed_state_dict[k.replace('bert.', 'distilbert.')] = v
            else:
                fixed_state_dict[k] = v

        model.load_state_dict(fixed_state_dict)
        model.eval()

        tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')

        return model, tokenizer, emotion_cols, thresholds

    def _load_mh_model(self, model_path: str):
        """Load mental health model"""
        print(f"Loading from: {model_path}")

        checkpoint = torch.load(model_path, map_location=self.device, weights_only=False)

        label_encoder = checkpoint['label_encoder']
        remaining_classes = checkpoint.get('remaining_classes', label_encoder.classes_)

        model = MHModel(len(label_encoder.classes_), len(self.emotion_cols) + 4).to(self.device)
        model.load_state_dict(checkpoint['model_state_dict'])
        model.eval()

        tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

        return model, tokenizer, label_encoder, remaining_classes

    def analyze_text(self, text: str, country_code: str = 'US') -> Dict:
        """Perform full analysis of text"""
        print(f"\nAnalyzing text: {text}")
        analysis = self._perform_analysis(text)
        suggestions = self._generate_gpt_suggestions(analysis, country_code)

        result = {
            **analysis,
            'suggestions': suggestions,
            'disclaimer': "These suggestions are generated by AI and are not medical advice. Please consult a healthcare professional for serious concerns."
        }

        # Add crisis resource if suicidal
        if analysis['mental_health_status'] == 'Suicidal':
            resource = self.crisis_resources.get(country_code, self.crisis_resources['default'])
            result['crisis_resource'] = f"🚨 CRISIS RESOURCE: {resource}"

        return result

    def _perform_analysis(self, text: str) -> Dict:
        """Run emotion and mental health analysis"""
        text = self._clean_text(text)

        # Emotion analysis
        emotion_results = self._analyze_emotions(text)

        # Mental health analysis
        mh_results = self._analyze_mental_health(text, emotion_results['emotion_probs'])

        return {
            'text': text,
            **emotion_results,
            **mh_results
        }

    def _clean_text(self, text: str) -> str:
        """Clean input text"""
        if not isinstance(text, str):
            return ""
        text = text.lower()
        text = re.sub(r'https?://\S+|www\.\S+', '', text)
        text = re.sub(r'<.*?>+', '', text)
        text = re.sub(r'[^\w\s]', '', text)
        text = re.sub(r'\s+', ' ', text).strip()
        return text

    def _analyze_emotions(self, text: str) -> Dict:
        """Perform emotion analysis"""
        print("\nRunning emotion analysis...")

        inputs = self.emotion_tokenizer(
            text,
            max_length=256,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt'
        )

        with torch.no_grad():
            outputs = self.emotion_model(
                inputs['input_ids'].to(self.device),
                inputs['attention_mask'].to(self.device)
            )
            probs = torch.sigmoid(outputs).cpu().numpy().flatten()

            print("\nEmotion Probabilities vs Thresholds:")
            for i, col in enumerate(self.emotion_cols):
                print(f"- {col}: {probs[i]:.4f} (threshold: {self.emotion_thresholds[i]:.4f}) {'✅' if probs[i] > self.emotion_thresholds[i] else '❌'}")

            preds = (probs > self.emotion_thresholds).astype(int)
            detected = [col for i, col in enumerate(self.emotion_cols) if preds[i] == 1]

            print(f"\nDetected emotions: {detected or 'None'}")

        return {
            'emotions': detected,
            'emotion_scores': {col: float(probs[i]) for i, col in enumerate(self.emotion_cols)},
            'emotion_probs': probs
        }

    def _analyze_mental_health(self, text: str, emotion_probs: np.ndarray) -> Dict:
        """Perform mental health analysis"""
        print("\nRunning mental health analysis...")

        inputs = self.mh_tokenizer(
            text,
            max_length=256,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt'
        )

        # Placeholder for VADER features
        vader_feats = np.array([0.0, 0.0, 0.0, 0.0])

        with torch.no_grad():
            features = torch.FloatTensor(
                np.concatenate([emotion_probs, vader_feats])
            ).unsqueeze(0).to(self.device)

            outputs = self.mh_model(
                inputs['input_ids'].to(self.device),
                inputs['attention_mask'].to(self.device),
                features
            )
            probs = torch.softmax(outputs, dim=1).cpu().numpy().flatten()
            pred = np.argmax(probs)

        status = self.label_encoder.inverse_transform([pred])[0]

        # Special handling for suicidal content
        suicide_keywords = ['end my life', 'kill myself', 'suicide', 'take my life']
        if any(keyword in text.lower() for keyword in suicide_keywords):
            status = 'Suicidal'
            probs = np.zeros_like(probs)
            probs[np.where(self.label_encoder.classes_ == 'Suicidal')[0][0]] = 0.99
            pred = np.argmax(probs)

        print("\nMental Health Probabilities:")
        max_len = max(len(label) for label in self.remaining_classes)
        for label, prob in zip(self.remaining_classes, probs):
            print(f"- {label.ljust(max_len)} : {prob:.4f} {'⬅' if label == status else ''}")

        print(f"\nMH Status: {status} (confidence: {probs[pred]:.2f})")

        return {
            'mental_health_status': status,
            'mental_health_confidence': float(probs[pred]),
            'mental_health_probabilities': {
                label: float(prob) for label, prob in zip(self.remaining_classes, probs)
            }
        }

    def _generate_gpt_suggestions(self, analysis: Dict, country_code: str) -> List[str]:
        """Generate suggestions using GPT-3.5"""
        print("\nGenerating suggestions...")

        prompt = f"""Based on this analysis, please provide 2-3 supportive suggestions:

        Text: "{analysis['text']}"

        Detected emotions: {', '.join(analysis['emotions']) or 'None'}
        Mental health status: {analysis['mental_health_status']} (confidence: {analysis['mental_health_confidence']:.2f})

        Please provide:
        1. Empathetic acknowledgment
        2. 2-3 practical suggestions
        3. Recommendation to consult a professional if needed
        """

        if analysis['mental_health_status'] in ['Suicidal', 'Depression']:
            resource = self.crisis_resources.get(country_code, self.crisis_resources['default'])
            prompt += f"\n\nIMPORTANT: The text suggests suicidal ideation. Please include this crisis resource prominently: {resource}"

        try:
            response = client.chat.completions.create(
                model=self.gpt_model,
                messages=[
                    {"role": "system", "content": self.system_prompt},
                    {"role": "user", "content": prompt}
                ],
                temperature=self.temperature,
                max_tokens=self.max_tokens
            )
            suggestions = [s.strip() for s in response.choices[0].message.content.split('\n') if s.strip()]

            # Ensure crisis resource is included for suicidal cases
            if analysis['mental_health_status'] == 'Suicidal':
                resource = self.crisis_resources.get(country_code, self.crisis_resources['default'])
                if not any(resource in s for s in suggestions):
                    suggestions.append(f"🚨 IMMEDIATE HELP: {resource}")

            return suggestions if suggestions else ["Consider reaching out to someone you trust."]
        except Exception as e:
            print(f"⚠️ Error generating suggestions: {str(e)}")
            return self._get_fallback_suggestions(analysis, country_code)

    def _get_fallback_suggestions(self, analysis: Dict, country_code: str) -> List[str]:
        """Provide fallback suggestions if GPT fails"""
        suggestions = []
        status = analysis['mental_health_status']

        if status == 'Suicidal':
            resource = self.crisis_resources.get(country_code, self.crisis_resources['default'])
            suggestions.append("You're not alone and your life matters. Please reach out for help immediately.")
            suggestions.append(f"🚨 CRISIS RESOURCE: {resource}")
            suggestions.append("Please contact emergency services or go to the nearest hospital if you're in immediate danger.")
        elif status in ['Depression']:
            suggestions.append("You're not alone. Consider reaching out to someone you trust.")
            resource = self.crisis_resources.get(country_code, self.crisis_resources['default'])
            suggestions.append(f"If in crisis, please contact: {resource}")
        elif status in ['Anxiety', 'Stress']:
            suggestions.append("Try deep breathing exercises to calm your nervous system.")
            suggestions.append("Writing down your thoughts might help process them.")
        else:
            suggestions.append("Maintain regular sleep, exercise, and social connections.")

        return suggestions

# ==============================================
# MAIN EXECUTION
# ==============================================

if __name__ == "__main__":
    try:
        print("\n🚀 Starting Mental Health Analysis System...")

        generator = GPT3SuggestionGenerator(
            emotion_model_path='best_emotion_model.pt',
            mh_model_path='best_mh_model.pt',
            gpt_model="gpt-3.5-turbo",
            temperature=0.7,
            max_tokens=200
        )

        sample_texts = [
            "I feel so anxious and scared all the time, I don't know what to do",
            "This is frustrating and annoying, everything keeps going wrong",
            "I'm feeling curious about this new opportunity that came up",
            "I've been feeling really down and hopeless lately",
            "I'm so happy with how things are going in my life right now",
            "I don't see the point in going on anymore, everything feels meaningless",
            "I've been thinking about ending my life. I can't take this pain anymore and I have a plan to do it"
        ]

        print("\n🔍 Sample Analysis with GPT-3.5 Suggestions:")
        for i, text in enumerate(sample_texts, 1):
            print(f"\n{'='*80}\nSample {i}:")
            try:
                result = generator.analyze_text(text)

                print("\n📊 Results:")
                print(f"- Text: {result['text']}")
                print(f"- Emotions: {', '.join(result['emotions']) or 'None'}")
                print(f"- MH Status: {result['mental_health_status']} (confidence: {result['mental_health_confidence']:.2f})")

                print("\nMental Health Probabilities:")
                max_len = max(len(label) for label in generator.remaining_classes)
                for label, prob in result['mental_health_probabilities'].items():
                    arrow = " ⬅" if label == result['mental_health_status'] else ""
                    print(f"  - {label.ljust(max_len)}: {prob:.4f}{arrow}")

                print("\n💡 Suggestions:")
                for j, suggestion in enumerate(result['suggestions'], 1):
                    print(f"  {j}. {suggestion}")

                if 'crisis_resource' in result:
                    print(f"\n{result['crisis_resource']}")

                print(f"\n⚠️ Disclaimer: {result['disclaimer']}")
            except Exception as e:
                print(f"\n❌ Error analyzing sample {i}: {str(e)}")
                continue

    except Exception as e:
        print(f"\n💥 Fatal error: {str(e)}")
        print("Troubleshooting:")
        print("1. Check model files exist")
        print("2. Verify OpenAI API key in .env")
        print("3. Ensure all packages are installed")
        print("4. Check internet connection")
    finally:
        print("\n✅ Analysis complete")


Using device: cuda

🚀 Starting Mental Health Analysis System...

Initializing models on device: cuda

Loading emotion model...
Loading from: best_emotion_model.pt
Found 13 emotion categories
Converting threshold dictionary to array...
✅ Emotion model loaded successfully
Detecting emotions: ['anger', 'annoyance', 'confusion', 'curiosity', 'desire', 'disappointment', 'disgust', 'embarrassment', 'fear', 'grief', 'nervousness', 'remorse', 'sadness']
Thresholds: [0.35 0.3  0.3 ]...

Loading mental health model...
Loading from: best_mh_model.pt
✅ Mental health model loaded successfully
Possible MH statuses: ['Anxiety' 'Bipolar' 'Depression' 'Normal' 'Personality disorder' 'Stress'
 'Suicidal']

🔍 Sample Analysis with GPT-3.5 Suggestions:

Sample 1:

Analyzing text: I feel so anxious and scared all the time, I don't know what to do

Running emotion analysis...

Emotion Probabilities vs Thresholds:
- anger: 0.0099 (threshold: 0.3500) ❌
- annoyance: 0.0177 (threshold: 0.3000) ❌
- confusion: 0.0